# 第 10 课 Notebook：具身智能安全扩展：停止、看门狗与故障恢复

**核心问题：怎样让取消信号最终变成可测量、可验证的物理停止？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**怎样让取消信号最终变成可测量、可验证的物理停止？**

In [ ]:
lesson_id = 10
essential_question = '怎样让取消信号最终变成可测量、可验证的物理停止？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`SafetySupervisor`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
# [Cell 7: 正常路径 — 具身安全底座与遥测校验]
try:
    from agent_course.courseware.public_examples import *
    from pathlib import Path
    safety_fixture = Path.cwd() / 'projects/capstones/warehouse_embodied_agent/fixtures/warehouse.json'
    if not safety_fixture.exists():
        raise FileNotFoundError
    safety_map = load_map(safety_fixture)
except Exception:
    # 零依赖 Colab / 云端环境内联教学底座
    from dataclasses import dataclass, field
    from enum import StrEnum
    from typing import Any
    from copy import deepcopy

    class StopReason(StrEnum):
        COMPLETED = "completed"
        UNSAFE_STATE = "unsafe_state"

    @dataclass(frozen=True)
    class Cell:
        x: int
        y: int

    @dataclass(frozen=True)
    class Waypoint:
        name: str
        cell: Cell

    @dataclass(frozen=True)
    class WarehouseMap:
        width: int
        height: int
        blocked: tuple[Cell, ...]
        dock: Waypoint
        waypoints: dict[str, Waypoint]
        safe_zone: tuple[Cell, ...] = (Cell(0, 0),)

    def load_map(path=None) -> WarehouseMap:
        dock = Waypoint("dock-a", Cell(0, 0))
        shelves = {"shelf-a": Waypoint("shelf-a", Cell(3, 1)), "dock-a": dock}
        return WarehouseMap(6, 4, (Cell(2, 1), Cell(2, 2)), dock, shelves, (Cell(0, 0),))

    @dataclass
    class RobotTelemetry:
        cell: Cell
        battery_percent: float = 100.0
        sensor_age_seconds: float = 0.1

        def model_copy(self, *, update: dict[str, Any] | None = None) -> 'RobotTelemetry':
            value = deepcopy(self)
            for key, item in (update or {}).items():
                setattr(value, key, item)
            return value

    class SafetyViolation(RuntimeError):
        pass

    class SafetySupervisor:
        def __init__(self, warehouse_map: WarehouseMap) -> None:
            self.warehouse_map = warehouse_map

        def validate_mission_start(self, telemetry: RobotTelemetry, *, approved: bool) -> None:
            if not approved:
                raise SafetyViolation("approval is required")
            self.validate_telemetry(telemetry)

        def validate_telemetry(self, telemetry: RobotTelemetry) -> None:
            if telemetry.sensor_age_seconds > 1:
                raise SafetyViolation("telemetry is stale")
            if telemetry.battery_percent <= 0:
                raise SafetyViolation("battery is empty")

    safety_map = load_map()

supervisor = SafetySupervisor(safety_map)
safe_telemetry = RobotTelemetry(cell=safety_map.safe_zone[0], sensor_age_seconds=0.1)
print({"map": f"{safety_map.width}x{safety_map.height}", "dock": safety_map.dock.name, "safe_telemetry": safe_telemetry})

## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
stale_telemetry = safe_telemetry.model_copy(update={"sensor_age_seconds": 5.0})
try:
    supervisor.validate_telemetry(stale_telemetry)
except SafetyViolation as error:
    stale_block_reason = str(error)
    print({"blocked": True, "reason": stale_block_reason})
else:
    raise AssertionError("stale telemetry must be blocked")

## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
# [Cell 13: 流程与终态可视化 (HTML Flowchart)]
from IPython.display import HTML, display

try:
    from agent_course.courseware.project_visuals import VisualStep, render_flow
except ImportError:
    import html
    from dataclasses import dataclass
    from typing import Sequence

    @dataclass(frozen=True, slots=True)
    class VisualStep:
        label: str
        detail: str
        status: str = "info"
        lane: str = "Runtime"

    def render_flow(*, title: str, question: str, steps: Sequence[VisualStep], takeaway: str) -> str:
        cards = []
        for index, step in enumerate(steps):
            arrow = '<div style="align-self:center;color:#2457c5;font-size:20px;font-weight:700">→</div>' if index else ""
            color_map = {"passed": "#276749", "blocked": "#9b2c2c", "info": "#2457c5"}
            bg_map = {"passed": "#e7f4ec", "blocked": "#f8e7e7", "info": "#e8eefb"}
            c = color_map.get(step.status, "#2457c5")
            bg = bg_map.get(step.status, "#e8eefb")
            cards.append(
                f'{arrow}<div style="min-width:140px;background:{bg};border:1px solid {c};border-top:4px solid {c};border-radius:8px;padding:10px;display:flex;flex-direction:column;gap:4px">'
                f'<span style="font-family:monospace;color:{c};font-size:11px;font-weight:700">{html.escape(step.lane)}</span>'
                f'<strong style="font-size:14px;color:#1f2933">{html.escape(step.label)}</strong>'
                f'<small style="color:#62707d;font-size:12px">{html.escape(step.detail)}</small></div>'
            )
        return (
            f'<div style="font-family:sans-serif;background:#f7f8fa;border:1px solid #cbd3da;border-radius:12px;padding:18px;margin:12px 0">'
            f'<h3 style="margin:0 0 6px">{html.escape(title)}</h3>'
            f'<p style="color:#62707d;font-size:13px;margin:0 0 12px"><strong>观察问题：</strong>{html.escape(question)}</p>'
            f'<div style="display:flex;gap:8px;overflow-x:auto;padding-bottom:8px">{"".join(cards)}</div>'
            f'<div style="border-left:4px solid #2457c5;padding:8px 12px;background:white;margin-top:12px;font-size:13px"><strong>读图结论：</strong>{html.escape(takeaway)}</div></div>'
        )

cancel_ack = True
velocity_zero = False
safe_zone = True
physical_stop_verified = cancel_ack and velocity_zero and safe_zone
safety_steps = (
    VisualStep("Cancel accepted", "ack=True", "passed", lane="Supervisor"),
    VisualStep("Velocity check", f"v={1.2 if not velocity_zero else 0.0}", "blocked", lane="Hardware"),
    VisualStep("Safe zone verify", f"zone={safe_zone}", "passed", lane="Supervisor"),
    VisualStep("Physical stop", f"verified={physical_stop_verified}", "blocked", lane="PhysicalWorld"),
)
display(
    HTML(
        render_flow(
            title="具身安全：取消信号 ≠ 物理停止",
            question="当取消确认已返回，为什么物理世界仍处于 unsafe？",
            steps=safety_steps,
            takeaway="取消只代表命令已接收；必须经由传感器证明速度归零且位于安全区，才算终态停止。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

具身安全里程碑：独立 Safety Supervisor、陈旧传感器、停止距离与仿真边界。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def stop_is_verified(cancel_ack: bool, velocity_zero: bool, safe_zone: bool) -> bool:
    # TODO: 不要把取消确认误当作已经停止。
    raise NotImplementedError

assert stop_is_verified(True, True, True)
assert not stop_is_verified(True, False, True)

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**怎样让取消信号最终变成可测量、可验证的物理停止？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。